In [1]:
# auto reload modules when they change
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from src import config
from src.data import compute_rate_constants, load_dataset, select_reactions
from src.kinetics import calculate_rate
from src.perturbation import run_reaction_pipeline_mode2, run_reaction_pipeline_mode2_parallel, run_reaction_pipeline_mode2_parallel2
from src.plotting import plot_kinetics_roh
from src.simulation import exponential_trajectory_points, run_simulation

sns.set_style("whitegrid")
colors = sns.color_palette("muted")
%matplotlib widget

In [3]:
from scipy import constants

hartree_energy_j = constants.physical_constants["Hartree energy"][0]
joules_per_kcal = constants.calorie * constants.kilo

HARTREE_TO_KCAL_MOL = hartree_energy_j * constants.Avogadro / joules_per_kcal

print(HARTREE_TO_KCAL_MOL)
print(f"Hartree to kcal/mol conversion factor: {HARTREE_TO_KCAL_MOL:.4f} kcal/mol per Hartree")

627.5094740628974
Hartree to kcal/mol conversion factor: 627.5095 kcal/mol per Hartree


In [4]:
# Global run controls
RUN_MODE = "mode2"  # "mode1" or "mode2"

# I/O

# Exchange Dataset (Several Reactions - GFN1-xTB Geometries)
DATASET_PATH = "imports/calculated-exchange.xlsx"
DATASET_SHEET = "raw energies"
EXPORT_DIR = Path("exports")
PLOTS_DIR = EXPORT_DIR / "batch-plots"
FORCE_DFT_CONDITIONS = True

# # DFT validation set
# DATASET_PATH = "imports/calc-dft.xlsx"
# DATASET_SHEET = "1M STATE FINAL"
# EXPORT_DIR = Path("exports")
# PLOTS_DIR = EXPORT_DIR / "batch-plots"

# Before starting a new run, clear the output directory to avoid confusion with old results
import shutil
if EXPORT_DIR.exists():
    confirm = input(f"Delete and recreate '{EXPORT_DIR}'? [y/N] ")
    if confirm.strip().lower() == "y":
        shutil.rmtree(EXPORT_DIR)
        EXPORT_DIR.mkdir()
        print(f"Recreated '{EXPORT_DIR}'.")
    else:
        print("Skipped clearing output directory.")
else:
    EXPORT_DIR.mkdir()

# Filtering
REACTION_FILTERS = {
    "imido": "im1",
    "roh": ["w1m", "d1m", "0", "w1p", "12", "9", "10", "d2m", "w2m", "8", "4", "1", "5"],
}

# Simulation controls
TEMPERATURE_C = config.TEMPERATURE_DEFAULT
SIMULATION_TIME = config.SIMULATION_TIME_DEFAULT
N_TIME_POINTS = config.TRAJECTORY_POINTS_DEFAULT
TIME_EXPONENT = config.TRAJECTORY_EXPONENT_DEFAULT
INITIAL_CONCENTRATIONS = dict(config.INITIAL_CONCENTRATIONS)

# Mode 2 controls
N_SAMPLES = config.PERTURBATION_N_SAMPLES_DEFAULT
SIGMA = config.PERTURBATION_SIGMA_DEFAULT
RANDOM_SEED = config.PERTURBATION_SEED_DEFAULT
REACTION_TIMEOUT_SECONDS = 1200  # skip reactions that take longer than this (seconds) for ALL SAMPLES simulations

# Optional plotting
SAVE_PLOTS = True
FIXED_PLOT_LIMITS = False

options = {
    "RUN_MODE": RUN_MODE,
    "DATASET_PATH": DATASET_PATH,
    "DATASET_SHEET": DATASET_SHEET,
    "EXPORT_DIR": str(EXPORT_DIR),
    "PLOTS_DIR": str(PLOTS_DIR),
    "REACTION_FILTERS": REACTION_FILTERS,
    "TEMPERATURE_C": TEMPERATURE_C,
    "SIMULATION_TIME": SIMULATION_TIME,
    "N_TIME_POINTS": N_TIME_POINTS,
    "TIME_EXPONENT": TIME_EXPONENT,
    "INITIAL_CONCENTRATIONS": INITIAL_CONCENTRATIONS,
    "N_SAMPLES": N_SAMPLES,
    "SIGMA": SIGMA,
    "RANDOM_SEED": RANDOM_SEED,
    "REACTION_TIMEOUT_SECONDS": REACTION_TIMEOUT_SECONDS,
    "SAVE_PLOTS": SAVE_PLOTS,
    "FIXED_PLOT_LIMITS": FIXED_PLOT_LIMITS,
}

print("Configured options:\n")
for key, value in options.items():
    print(f"- {key}: {value}")

Skipped clearing output directory.
Configured options:

- RUN_MODE: mode2
- DATASET_PATH: imports/calculated-exchange.xlsx
- DATASET_SHEET: raw energies
- EXPORT_DIR: exports
- PLOTS_DIR: exports/batch-plots
- REACTION_FILTERS: {'imido': 'im1', 'roh': ['w1m', 'd1m', '0', 'w1p', '12', '9', '10', 'd2m', 'w2m', '8', '4', '1', '5']}
- TEMPERATURE_C: 70
- SIMULATION_TIME: 43200
- N_TIME_POINTS: 1000
- TIME_EXPONENT: 3
- INITIAL_CONCENTRATIONS: {'bispyr': 0.1, 'roh': 0.2, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0}
- N_SAMPLES: 1000
- SIGMA: 3.0
- RANDOM_SEED: 42
- REACTION_TIMEOUT_SECONDS: 1200
- SAVE_PLOTS: True
- FIXED_PLOT_LIMITS: False


In [5]:
if DATASET_PATH == "imports/calc-dft.xlsx" or FORCE_DFT_CONDITIONS:
    # List of labels in the DFT dataset (for reference):
    dft_labels = [
        "im1-ad1",
        "im1-oc1",
        "im1-oc3",
        "im2-bitet0",
        "im2-oc0",
        "im2-oc1",
        "im2-oc3",
        "im3-15",
        "im3-26",
        "im3-29",
        "im3-31",
        "im4-oc1",
        "im5-oc1",
        "im5-oc2",
        "im5-oc3",
        "im5-oc4",
    ]

    # Custom settings for the DFT dataset per reaction label
    DFT_DATASET_SETTINGS = {
        # Entry 1 (46)
        'im5-oc1': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.02, 'roh': 0.02, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 2 (43)
        'im5-oc2': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.02, 'roh': 0.02, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 3 (44)
        'im5-oc3': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.02, 'roh': 0.02, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 4 (45)
        'im5-oc4': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.02, 'roh': 0.02, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 5 (49)
        'im2-oc0': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.01, 'roh': 0.01, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 6 (48)
        'im2-oc1': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.01, 'roh': 0.01, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 7 (72)
        'im2-oc3': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.02, 'roh': 0.02, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 8 (38) --- removed
        # Entry 9A (50)
        'im4-oc1-1eq': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.01, 'roh': 0.01, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 9B (50)
        'im4-oc1-2eq': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.02, 'roh': 0.04, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 10 (40)
        'im1-oc1': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.02, 'roh': 0.02, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 11 (21)
        'im1-oc3': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.02, 'roh': 0.02, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 12 (24)
        'im1-ad1': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.02, 'roh': 0.02, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        # Entry 13 (71)
        'im3-j15': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.1, 'roh': 0.2, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 6,
            'TEMPERATURE_C': 22
        },
        # Entry 14A (68)
        'im3-j29-1eq': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.03, 'roh': 0.03, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 2,
            'TEMPERATURE_C': 22
        },
        # Entry 14B (68)
        'im3-j29-2eq': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.1, 'roh': 0.2, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 12,
            'TEMPERATURE_C': 70
        },
        # Entry 15 (70)
        'im3-j31': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.1, 'roh': 0.2, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 12,
            'TEMPERATURE_C': 70
        },
        # Entry 16 (64)
        'im3-j26': {
            'INITIAL_CONCENTRATIONS': {'bispyr': 0.1, 'roh': 0.1, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0},
            'SIMULATION_TIME': 3600 * 1,
            'TEMPERATURE_C': 22
        },
        
    }
    DFT_SELECTED_REACTION = 'im5-oc2'

    # Apply custom settings for the selected reaction to the global options:
    if DFT_SELECTED_REACTION in DFT_DATASET_SETTINGS:
        custom_settings = DFT_DATASET_SETTINGS[DFT_SELECTED_REACTION]

        # # Set sigma to 2 kcal/mol (DFT Only)
        # options.update(
        #     {
        #         'SIGMA': 2.0
        #     }
        # )

        # Set sigma to 3 kcal/mol (XTB Only)
        options.update(
            {
                'SIGMA': 3.0
            }
        )

        options.update(custom_settings)

    print("Configured options:\n")
    for key, value in options.items():
        print(f"- {key}: {value}")

    # Rebind actual runtime variables from options
    INITIAL_CONCENTRATIONS = options["INITIAL_CONCENTRATIONS"]
    SIMULATION_TIME = options["SIMULATION_TIME"]
    TEMPERATURE_C = options["TEMPERATURE_C"]
    SIGMA = options["SIGMA"]
else:
    print("Using default settings for the selected dataset (not DFT validation set).")

Configured options:

- RUN_MODE: mode2
- DATASET_PATH: imports/calculated-exchange.xlsx
- DATASET_SHEET: raw energies
- EXPORT_DIR: exports
- PLOTS_DIR: exports/batch-plots
- REACTION_FILTERS: {'imido': 'im1', 'roh': ['w1m', 'd1m', '0', 'w1p', '12', '9', '10', 'd2m', 'w2m', '8', '4', '1', '5']}
- TEMPERATURE_C: 22
- SIMULATION_TIME: 3600
- N_TIME_POINTS: 1000
- TIME_EXPONENT: 3
- INITIAL_CONCENTRATIONS: {'bispyr': 0.02, 'roh': 0.02, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0}
- N_SAMPLES: 1000
- SIGMA: 3.0
- RANDOM_SEED: 42
- REACTION_TIMEOUT_SECONDS: 1200
- SAVE_PLOTS: True
- FIXED_PLOT_LIMITS: False


In [6]:
def calculate_mechanism_roh(energies:pd.Series):
    reactants = energies.get('bispyrrolide', np.nan) + energies.get('roh', np.nan) + energies.get('roh', np.nan)
    parsed_energies = {
        "reactants": reactants,
        "a1": energies.get("a1", np.nan) + energies.get('roh', np.nan),
        "ts1": energies.get("ts1", np.nan) + energies.get('roh', np.nan),
        "map": energies.get("map", np.nan) + energies.get('roh', np.nan) + energies.get('pyrrole', np.nan),
        "a2": energies.get("a2", np.nan) + energies.get('pyrrole', np.nan),
        "ts2": energies.get("ts2", np.nan) + energies.get('pyrrole', np.nan),
        "bis": energies.get("bis", np.nan) + energies.get('pyrrole', np.nan) + energies.get('pyrrole', np.nan),
    }
    parsed_energies = pd.Series(parsed_energies)
    return parsed_energies

def calculate_barriers_roh(energies:pd.Series):
    b1 = energies['ts1'] - energies['reactants']
    b2 = energies['ts2'] - energies['map']
    b3 = energies['ts1'] - energies['map']
    b4 = energies['ts2'] - energies['bis']
    return pd.Series({'b1':b1, 'b2':b2, 'b3':b3, 'b4':b4})

calculate_mechanism = calculate_mechanism_roh
calculate_barriers = calculate_barriers_roh

In [7]:
dataset = load_dataset(DATASET_PATH, sheet_name=DATASET_SHEET)

if FORCE_DFT_CONDITIONS:
    dataset = dataset.loc[dft_labels]

# Replace exact zeros by np.nan
dataset.replace(0, np.nan, inplace=True)
display(dataset)

,bispyrrolide,roh,pyrrole,map,bis,ts1,ts2,a1,a2
im1-ad1,-1555.214453,-538.586163,-288.821797,-1804.996935,-2054.779851,-2093.778114,-2343.562156,-2093.791002,-2343.576777
im1-oc1,-1555.214453,-1651.041717,-288.821797,-2917.454643,-4279.690504,-3206.233025,-4568.467377,-3206.240500,NaN
im1-oc3,-1555.214453,-6599.495949,-288.821797,-7865.906592,-14176.583860,-8154.679900,-14465.355582,-8154.690794,NaN
im2-bitet0,-1397.999596,-1447.753129,-288.821797,-2556.947631,-3715.889411,-2845.728054,-4004.679732,-2845.740751,-4004.686950
im2-oc0,-1397.999596,-1452.508861,-288.821797,-2561.700798,-3725.400104,-2850.482440,-4014.186108,-2850.497326,-4014.203281
im2-oc1,-1397.999596,-1651.041717,-288.821797,-2760.238553,-4122.474457,-3049.017266,-4411.254608,-3049.024470,-4411.270404
im2-oc3,-1397.999596,-6599.495949,-288.821797,-7708.689892,-14019.380905,-7997.466649,-14308.143093,-7997.478766,-14308.178948
im3-15,-1815.695209,-769.655929,-288.821797,-2296.546311,-2777.398290,-2585.319374,-3066.162156,-2585.328077,-3066.180763
im3-26,-1815.695209,-1005.510019,-288.821797,-2532.402218,-3249.093368,-2821.164059,-3537.847611,-2821.176634,NaN
im3-29,-1815.695209,-1231.796937,-288.821797,-2758.690752,-3701.678679,-3047.455183,-3990.448529,-3047.468724,-3990.456735


In [8]:
# For mode2 we drop the columns a1 and a2 to avoid issues with missing values
_dataset = dataset.drop(columns=['a1','a2'])

# Show which entries have at least one missing value after filtering (NaN)
missing_data_entries = _dataset[_dataset.isna().any(axis=1)]
print("Entries with missing data after removing a1 and a2:")
display(missing_data_entries)

# Entries that have neither TS1 nor TS2 energies are not suitable for mode 2, so we can filter them out
print("Entries with missing TS1 or TS2 energies are dropped for mode 2")
_dataset = _dataset.dropna(subset=['ts1', 'ts2'])

# Show missing data status after filtering
missing_data_entries_after = _dataset[_dataset.isna().any(axis=1)]
print("Entries with missing data after filtering:")
display(missing_data_entries_after)

dataset = _dataset

Entries with missing data after removing a1 and a2:


,bispyrrolide,roh,pyrrole,map,bis,ts1,ts2


Entries with missing TS1 or TS2 energies are dropped for mode 2
Entries with missing data after filtering:


,bispyrrolide,roh,pyrrole,map,bis,ts1,ts2


In [9]:
mechanism_df = dataset.T.apply(calculate_mechanism)

mechanism_df -= mechanism_df.loc['reactants']  # Set reactants as zero reference
mechanism_df *= HARTREE_TO_KCAL_MOL  # Convert from Hartree to kcal/mol
mechanism_df = mechanism_df.T

print('-'*75)
print("Calculated mechanism energies (kcal/mol):")
print('-'*75)
display(mechanism_df.round(1))

barriers_df = mechanism_df.T.apply(calculate_barriers).T
print('-'*75)
print("Calculated barrier energies (kcal/mol):")
print('-'*75)
display(barriers_df.round(1))

# Convert barriers -> rate constants using phase-1 utilities
rates_df = compute_rate_constants(barriers_df, temperature=TEMPERATURE_C)
print('-'*75)
print("Calculated rate constants (1/s):")
print('-'*75)
display(rates_df.head().style.format("{:.2e}"))

---------------------------------------------------------------------------
Calculated mechanism energies (kcal/mol):
---------------------------------------------------------------------------


,reactants,a1,ts1,map,a2,ts2,bis
im1-ad1,0.0,NaN,14.1,-11.4,NaN,1.8,-23.0
im1-oc1,0.0,NaN,14.5,-12.7,NaN,5.5,-22.7
im1-oc3,0.0,NaN,19.1,-11.3,NaN,18.2,-13.2
im2-bitet0,0.0,NaN,15.5,-10.5,NaN,2.7,-17.0
im2-oc0,0.0,NaN,16.3,-8.9,NaN,5.9,-16.6
im2-oc1,0.0,NaN,15.1,-11.9,NaN,4.2,-22.0
im2-oc3,0.0,NaN,18.1,-10.1,NaN,16.7,-20.7
im3-15,0.0,NaN,19.9,-10.6,NaN,14.5,-21.8
im3-26,0.0,NaN,25.8,-11.8,NaN,28.8,-13.6
im3-29,0.0,NaN,23.2,-12.8,NaN,11.8,-20.8


---------------------------------------------------------------------------
Calculated barrier energies (kcal/mol):
---------------------------------------------------------------------------


,b1,b2,b3,b4
im1-ad1,14.1,13.1,25.5,24.8
im1-oc1,14.5,18.2,27.2,28.2
im1-oc3,19.1,29.5,30.4,31.4
im2-bitet0,15.5,13.2,26.0,19.8
im2-oc0,16.3,14.8,25.2,22.5
im2-oc1,15.1,16.1,27.0,26.1
im2-oc3,18.1,26.8,28.3,37.4
im3-15,19.9,25.2,30.6,36.4
im3-26,25.8,40.6,37.6,42.4
im3-29,23.2,24.6,36.0,32.6


---------------------------------------------------------------------------
Calculated rate constants (1/s):
---------------------------------------------------------------------------


,k1d,k1r,k2d,k2r
im1-ad1,2.15e+02,8.23e-07,1.14e+03,2.75e-06
im1-oc1,1.08e+02,4.12e-08,2.10e-01,8.21e-09
im1-oc3,4.13e-02,1.81e-10,9.31e-10,3.32e-11
im2-bitet0,2.11e+01,3.66e-07,1.04e+03,1.45e-02
im2-oc0,5.01e+00,1.35e-06,7.01e+01,1.44e-04


In [10]:
# Apply the REACTION_FILTERS global option to select a subset of reactions for simulation
# Currently not in use
# selected_rates = select_reactions(rates_df, REACTION_FILTERS)

# Otherwise, check the available reactions and select one by index for testing
print("Available reactions:")
for i, idx in enumerate(rates_df.index):
    print(f"  {i}: {idx}")

# Select one reaction by index
selected_rates = rates_df.copy()
# selected_rates = selected_rates.iloc[[0,8,38,39,43,69,71,77,78,85,89,91,104,111,133]]
# selected_rates = selected_rates.iloc[[96,129]]
# selected_rates = selected_rates.iloc[5:]

display(selected_rates)

if DATASET_PATH == "imports/calc-dft.xlsx" or FORCE_DFT_CONDITIONS:
    # For the DFT dataset, we select the reaction specified in DFT_SELECTED_REACTION

    # Count the number of - in the label, if >1 do rsplit with n-1 and keep only 0 (ie, im4-oc1-2eq should be treated as im4-oc1)
    shifter_counter = DFT_SELECTED_REACTION.count('-')
    if shifter_counter > 1:
        base_label = DFT_SELECTED_REACTION.rsplit('-', shifter_counter-1)[0]
        print(f'WARNING: The selected DFT reaction label "{DFT_SELECTED_REACTION}" contains {shifter_counter} dashes. Using base label "{base_label}" for filtering.')
    else:
        base_label = DFT_SELECTED_REACTION
        print(f'Selected DFT reaction label "{DFT_SELECTED_REACTION}" contains {shifter_counter} dashes. Using it directly for filtering.')

    # Remove j from base_label to match the format in the dataset (eg, im3-j15 should be im3-15)
    if 'j' in base_label:
        base_label = base_label.replace('j', '')
        print(f'Adjusted base label to "{base_label}" for matching dataset format.')

    selected_rates = selected_rates.loc[[base_label]]

print('-'*75)
print("Selected reactions:")
print('-'*75)
print(f"Number of selected reactions: {len(selected_rates)}")

print('-'*75)
print("Barrier energies for selected reactions (kcal/mol):")
print('-'*75)
display(barriers_df.loc[selected_rates.index].round(1))

print('-'*75)
print("Rate constants for selected reactions (1/s):")
print('-'*75)
display(selected_rates.head())


Available reactions:
  0: im1-ad1
  1: im1-oc1
  2: im1-oc3
  3: im2-bitet0
  4: im2-oc0
  5: im2-oc1
  6: im2-oc3
  7: im3-15
  8: im3-26
  9: im3-29
  10: im3-31
  11: im4-oc1
  12: im5-oc1
  13: im5-oc2
  14: im5-oc3
  15: im5-oc4


,k1d,k1r,k2d,k2r
im1-ad1,2.153436e+02,8.225911e-07,1.143284e+03,2.745445e-06
im1-oc1,1.081448e+02,4.120981e-08,2.096341e-01,8.211809e-09
im1-oc3,4.125048e-02,1.809535e-10,9.307581e-10,3.316624e-11
im2-bitet0,2.112333e+01,3.658900e-07,1.040722e+03,1.454456e-02
im2-oc0,5.009515e+00,1.348996e-06,7.007764e+01,1.435491e-04
im2-oc1,4.122697e+01,5.875739e-08,7.324050e+00,2.736495e-07
im2-oc3,2.302866e-01,7.252124e-09,8.424119e-08,1.232607e-15
im3-15,1.068575e-02,1.391469e-10,1.455860e-06,7.421066e-15
im3-26,4.557256e-07,8.500482e-16,5.749342e-18,2.505689e-19
im3-29,4.101319e-05,1.356920e-14,3.910671e-06,4.473093e-12


Selected DFT reaction label "im5-oc2" contains 1 dashes. Using it directly for filtering.
---------------------------------------------------------------------------
Selected reactions:
---------------------------------------------------------------------------
Number of selected reactions: 1
---------------------------------------------------------------------------
Barrier energies for selected reactions (kcal/mol):
---------------------------------------------------------------------------


,b1,b2,b3,b4
im5-oc2,24.0,15.4,33.9,28.3


---------------------------------------------------------------------------
Rate constants for selected reactions (1/s):
---------------------------------------------------------------------------


,k1d,k1r,k2d,k2r
im5-oc2,0.00001,4.801363e-13,24.184518,6.546855e-09


In [11]:
# Quick run reminder:
# 1) Set RUN_MODE = "mode1" for baseline from rates.
# 2) Set RUN_MODE = "mode2" and implement mechanism_energies_for_mode2 + run_single_pipeline_mode2.

print(f"Configured run mode: {RUN_MODE}")
print(f"Selected reactions: {len(selected_rates)}")
print("Ready to execute.")

Configured run mode: mode2
Selected reactions: 1
Ready to execute.


In [12]:
def simulate_from_rates(reaction_rates: pd.Series) -> tuple[pd.DataFrame, dict[str, np.ndarray], dict[str, np.ndarray], dict[str, np.ndarray]]:
    points = exponential_trajectory_points(SIMULATION_TIME, num_points=N_TIME_POINTS, exponent=TIME_EXPONENT)
    simulation = run_simulation(reaction_rates.to_dict(), INITIAL_CONCENTRATIONS, points)

    concentrations = {
        label: simulation["y"][idx]
        for idx, label in enumerate(config.ENTITIES)
    }

    c0 = concentrations["bispyr"][0]
    yields = {k: (v / c0) for k, v in concentrations.items()}
    conversions = {k: (1 - v) for k, v in yields.items()}
    conversions["roh"] = 1 - (concentrations["roh"] / concentrations["roh"][0])

    # NOTICE:
    # We use the yields (not concentrations) for the trajectory DataFrame, to have a common scale (concentration independent scale)
    # But it is equivalent to use concentrations, as they are just scaled by a constant factor (initial concentration of bispyr). 
    trajectory = pd.concat(
        [pd.Series(simulation["t"], name="time"), pd.DataFrame(yields)],
        axis=1,
    )
    return trajectory, concentrations, yields, conversions

In [13]:
# Outputs for Mode 1
concentrations_all: dict[str, dict[str, np.ndarray]] = {}
yields_all: dict[str, dict[str, np.ndarray]] = {}
conversions_all: dict[str, dict[str, np.ndarray]] = {}
trajectories_all: dict[str, pd.DataFrame] = {}
summary_rows: list[dict[str, float | str]] = []

if RUN_MODE == "mode1":
    if SAVE_PLOTS:
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)

    for reaction_label, reaction_rates in selected_rates.iterrows():
        print(f"Current reaction: {reaction_label}")

        if reaction_rates.isna().any():
            print("At least one rate constant for this reaction is missing. Skipping simulation")
            continue

        rate_threshold = calculate_rate(25, TEMPERATURE_C)
        if reaction_rates["k1d"] <= rate_threshold:
            print("Warning: 1st exchange may be inaccessible")

        trajectory, concentrations, yields, conversions = simulate_from_rates(reaction_rates)

        xt = len(trajectory) - 1
        summary_rows.append(
            {
                "reaction": reaction_label,
                "conv_bispyr_final": float(conversions["bispyr"][xt]),
                "conv_roh_final": float(conversions["roh"][xt]),
                "yield_map_final": float(yields["map"][xt]),
                "yield_bis_final": float(yields["bis"][xt]),
            }
        )

        concentrations_all[reaction_label] = concentrations
        yields_all[reaction_label] = yields
        conversions_all[reaction_label] = conversions
        trajectories_all[reaction_label] = trajectory

        if SAVE_PLOTS:
            plot_kinetics_roh(
                times=trajectory["time"].to_numpy(),
                concentrations=concentrations,
                reaction_label=reaction_label,
                colors=colors,
                set_ylim=FIXED_PLOT_LIMITS,
                set_xlim=FIXED_PLOT_LIMITS,
                save_path=str(PLOTS_DIR),
            )

    mode1_summary = pd.DataFrame(summary_rows)
    display(mode1_summary.head())
else:
    print("Mode 1 not selected. Skipping this cell")

Mode 1 not selected. Skipping this cell


In [14]:
import signal
from contextlib import contextmanager

@contextmanager
def _reaction_timeout(seconds: int, label: str):
    """Raise TimeoutError if the block takes longer than `seconds`."""
    def _handler(signum, frame):
        raise TimeoutError(f"{label} exceeded {seconds}s timeout")
    old_handler = signal.signal(signal.SIGALRM, _handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

def run_single_pipeline_mode2(perturbed_mechanism: pd.Series) -> pd.DataFrame:
    """Run one deterministic pipeline instance for a perturbed mechanism series."""
    barriers = calculate_barriers(perturbed_mechanism)
    rates_one = compute_rate_constants(pd.DataFrame([barriers]), temperature=TEMPERATURE_C).iloc[0]
    trajectory, concentrations, yields, conversions = simulate_from_rates(rates_one)
    return trajectory

mode2_results: dict[str, dict[str, object]] = {}
mode2_skipped: list[str] = []

if RUN_MODE == "mode2":
    for reaction_label in tqdm(selected_rates.index, desc="Mode 2 reactions"):
        print(f"Current reaction: {reaction_label}")

        baseline_mechanism = mechanism_df.loc[reaction_label]
        try:
            with _reaction_timeout(REACTION_TIMEOUT_SECONDS, reaction_label):
                mode2_results[reaction_label] = run_reaction_pipeline_mode2_parallel2(
                    baseline_energies=baseline_mechanism,
                    run_single_pipeline=run_single_pipeline_mode2,
                    n_samples=1000,
                    sigma=SIGMA,
                    random_seed=RANDOM_SEED,
                    aggregate=False,
                    export_trajectories=True,
                    export_dir=EXPORT_DIR,
                    reaction_label=reaction_label,
                )
        except TimeoutError as e:
            print(f"Warning: {e} — skipping")
            mode2_skipped.append(reaction_label)

    print(f"Mode 2 completed for {len(mode2_results)} reactions ({len(mode2_skipped)} skipped due to timeout)")

Mode 2 reactions:   0%|          | 0/1 [00:00<?, ?it/s]

Current reaction: im5-oc2
Mode 2 completed for 1 reactions (0 skipped due to timeout)


In [15]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_MODE == "mode1":
    mode1_summary.to_csv(EXPORT_DIR / "mode1_summary.csv", index=False)

    # Optional trajectory export (long format)
    mode1_long = []
    for reaction_label, traj_df in trajectories_all.items():
        tmp = traj_df.copy()
        tmp.insert(0, "reaction", reaction_label)
        mode1_long.append(tmp)

    if mode1_long:
        pd.concat(mode1_long, ignore_index=True).to_csv(EXPORT_DIR / "mode1_trajectories.csv", index=False)

    print("Exported Mode 1 summary and trajectories")

if RUN_MODE == "mode2":
    for reaction_label, result in tqdm(mode2_results.items(), desc="Mode 2 exports", total=len(mode2_results)):
        result["samples"].to_csv(EXPORT_DIR / f"{reaction_label}_mode2_samples.csv", index=False)

        # If the 'timed_out' key is present, export the timed out list to a txt file
        if result.get("timed_out"):
            timed_out_list = result["timed_out"]
            with open(EXPORT_DIR / f"{reaction_label}_mode2_timed_out.txt", "w") as f:
                for item in timed_out_list:
                    f.write(f"{item}\n")
                    
        # Avoid duplicate writes: if core pipeline already exported perturbed energies,
        # do not export the same table again from this notebook cell.
        if "perturbed_energies_export_path" not in result:
            perturbed_df = result.get("perturbed_energies")
            if perturbed_df is not None:
                perturbed_df.to_csv(
                    EXPORT_DIR / f"{reaction_label}_mode2_perturbed_energies.csv", index=False
                )
            else:
                print(
                    f"Warning: perturbed_energies missing for {reaction_label}. "
                    "Re-run the Mode 2 cell to regenerate full perturbation tracking."
                )

        agg = result.get("aggregate")
        if agg:
            agg["mean"].to_csv(EXPORT_DIR / f"{reaction_label}_mode2_mean.csv", index=False)
            agg["std"].to_csv(EXPORT_DIR / f"{reaction_label}_mode2_std.csv", index=False)
            agg["percentiles"].to_csv(EXPORT_DIR / f"{reaction_label}_mode2_percentiles.csv", index=False)

        if "trajectories_export_path" in result:
            print(f"Trajectories exported: {result['trajectories_export_path']}")
        if "perturbed_energies_export_path" in result:
            print(f"Perturbed energies exported: {result['perturbed_energies_export_path']}")

    print("Exported Mode 2 samples, perturbed energies, and aggregate tables")

Mode 2 exports:   0%|          | 0/1 [00:00<?, ?it/s]

Trajectories exported: exports/im5-oc2_mode2_trajectories.pkl.gz
Perturbed energies exported: exports/im5-oc2_mode2_perturbed_energies.csv
Exported Mode 2 samples, perturbed energies, and aggregate tables
